<a href="https://colab.research.google.com/github/bojannithya-tech/nithyabojan/blob/main/AI_POS_Transaction_Anomaly_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 1: Synthetic POS Transaction Dataset

Part 1A — Create the first 3 POS transactions

In [9]:
# Synthetic POS transaction dataset
# We start with 3 scenarios:
# 1. Normal cashback
# 2. Higher-than-expected cashback
# 3. Negative cashback

transactions = [
    {
        "transaction_id": "TXN001",
        "store_id": "STORE101",
        "lane_id": "LANE01",

        "items": [
            {"item_id": "ITEM001", "price": 20.00, "quantity": 2},
            {"item_id": "ITEM002", "price": 10.00, "quantity": 1}
        ],

        "basket_total": 50.00,

        "promotion": {
            "discount_amount": 5.00
        },

        "gift_card": {
            "used": False,
            "amount": 0.00,
            "activation_status": "NOT_APPLICABLE",
            "failure_amount": 0.00
        },

        "payment": {
            "type": "DEBIT",
            "amount": 45.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": 5.00
        }
    },

    {
        "transaction_id": "TXN002",
        "store_id": "STORE101",
        "lane_id": "LANE02",

        "items": [
            {"item_id": "ITEM101", "price": 40.00, "quantity": 1},
            {"item_id": "ITEM102", "price": 30.00, "quantity": 1}
        ],

        "basket_total": 70.00,

        "promotion": {
            "discount_amount": 5.00
        },

        "gift_card": {
            "used": True,
            "amount": 25.00,
            "activation_status": "FAILED",
            "failure_amount": 25.00
        },

        "payment": {
            "type": "CREDIT",
            "amount": 70.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": 30.00
        }
    },

    {
        "transaction_id": "TXN003",
        "store_id": "STORE102",
        "lane_id": "LANE03",

        "items": [
            {"item_id": "ITEM201", "price": 50.00, "quantity": 1},
            {"item_id": "ITEM202", "price": 20.00, "quantity": 1}
        ],

        "basket_total": 70.00,

        "promotion": {
            "discount_amount": 10.00
        },

        "gift_card": {
            "used": False,
            "amount": 0.00,
            "activation_status": "NOT_APPLICABLE",
            "failure_amount": 0.00
        },

        "payment": {
            "type": "DEBIT",
            "amount": 60.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": -15.00
        }
    }
]

print("Synthetic transactions created:", len(transactions))

Synthetic transactions created: 3


Part 1B — Display the transactions

In [10]:
import json

for transaction in transactions:
    print("=" * 60)
    print(json.dumps(transaction, indent=2))

{
  "transaction_id": "TXN001",
  "store_id": "STORE101",
  "lane_id": "LANE01",
  "items": [
    {
      "item_id": "ITEM001",
      "price": 20.0,
      "quantity": 2
    },
    {
      "item_id": "ITEM002",
      "price": 10.0,
      "quantity": 1
    }
  ],
  "basket_total": 50.0,
  "promotion": {
    "discount_amount": 5.0
  },
  "gift_card": {
    "used": false,
    "amount": 0.0,
    "activation_status": "NOT_APPLICABLE",
    "failure_amount": 0.0
  },
  "payment": {
    "type": "DEBIT",
    "amount": 45.0,
    "status": "SUCCESS"
  },
  "cashback": {
    "expected": 5.0,
    "actual": 5.0
  }
}
{
  "transaction_id": "TXN002",
  "store_id": "STORE101",
  "lane_id": "LANE02",
  "items": [
    {
      "item_id": "ITEM101",
      "price": 40.0,
      "quantity": 1
    },
    {
      "item_id": "ITEM102",
      "price": 30.0,
      "quantity": 1
    }
  ],
  "basket_total": 70.0,
  "promotion": {
    "discount_amount": 5.0
  },
  "gift_card": {
    "used": true,
    "amount": 25.0,


Part 2 — Build the Transaction Health Check

## Part 2: Transaction Health Check

The Transaction Health Check performs deterministic validation
before invoking GenAI agents.

It classifies a completed POS transaction as:

- NORMAL
- HIGH_CASHBACK
- NEGATIVE_CASHBACK

Financial calculations are performed using deterministic Python
logic rather than relying on an LLM.

In [11]:
def check_transaction_health(transaction):

    expected = transaction["cashback"]["expected"]
    actual = transaction["cashback"]["actual"]

    variance = round(actual - expected, 2)

    # Negative cashback has highest priority
    if actual < 0:
        status = "ANOMALY"
        anomaly_type = "NEGATIVE_CASHBACK"

    # Cashback greater than expected
    elif actual > expected:
        status = "ANOMALY"
        anomaly_type = "HIGH_CASHBACK"

    else:
        status = "NORMAL"
        anomaly_type = "NONE"

    return {
        "transaction_id": transaction["transaction_id"],
        "status": status,
        "anomaly_type": anomaly_type,
        "expected_cashback": expected,
        "actual_cashback": actual,
        "variance": variance
    }

Part 2B — Test the Health Check

In [12]:
health_results = []

for transaction in transactions:

    result = check_transaction_health(transaction)
    health_results.append(result)

    print("=" * 50)
    print("Transaction ID :", result["transaction_id"])
    print("Status         :", result["status"])
    print("Anomaly Type   :", result["anomaly_type"])
    print("Expected       : $", result["expected_cashback"])
    print("Actual         : $", result["actual_cashback"])
    print("Variance       : $", result["variance"])

Transaction ID : TXN001
Status         : NORMAL
Anomaly Type   : NONE
Expected       : $ 5.0
Actual         : $ 5.0
Variance       : $ 0.0
Transaction ID : TXN002
Status         : ANOMALY
Anomaly Type   : HIGH_CASHBACK
Expected       : $ 5.0
Actual         : $ 30.0
Variance       : $ 25.0
Transaction ID : TXN003
Status         : ANOMALY
Anomaly Type   : NEGATIVE_CASHBACK
Expected       : $ 5.0
Actual         : $ -15.0
Variance       : $ -20.0


Part 3 — Pattern & Correlation Analysis

## Part 3: Pattern and Correlation Analysis

For anomalous transactions, the system investigates whether the
cashback variance correlates with other monetary values in the
transaction.

The analysis checks relationships with:

- Gift card amount
- Gift card failure amount
- Promotion/discount amount
- Payment amount
- Individual item amounts

The detected correlations are treated as investigation evidence,
not as confirmed root causes.

Create the correlation tool

In [13]:
def analyze_amount_correlations(transaction, health_result):

    # Normal transactions don't require deeper investigation
    if health_result["status"] == "NORMAL":
        return {
            "transaction_id": transaction["transaction_id"],
            "correlations": [],
            "message": "No anomaly detected. Correlation analysis not required."
        }

    expected = health_result["expected_cashback"]
    actual = health_result["actual_cashback"]

    # For HIGH cashback, investigate the excess.
    # For NEGATIVE cashback, investigate the magnitude of the negative value.
    if health_result["anomaly_type"] == "HIGH_CASHBACK":
        suspicious_amount = round(actual - expected, 2)
        investigation_basis = "EXCESS_CASHBACK"

    elif health_result["anomaly_type"] == "NEGATIVE_CASHBACK":
        suspicious_amount = round(abs(actual), 2)
        investigation_basis = "NEGATIVE_CASHBACK_MAGNITUDE"

    else:
        suspicious_amount = 0
        investigation_basis = "NONE"

    correlations = []

    # Candidate transaction values
    candidates = {
        "gift_card_amount": transaction["gift_card"]["amount"],
        "gift_card_failure_amount": transaction["gift_card"]["failure_amount"],
        "promotion_discount": transaction["promotion"]["discount_amount"],
        "payment_amount": transaction["payment"]["amount"]
    }

    # Compare suspicious amount with transaction-level values
    for name, value in candidates.items():
        if value > 0 and abs(suspicious_amount - value) < 0.01:
            correlations.append({
                "type": "EXACT_MATCH",
                "field": name,
                "value": value
            })

    # Compare against individual item amounts
    for item in transaction["items"]:

        item_total = round(item["price"] * item["quantity"], 2)

        if abs(suspicious_amount - item_total) < 0.01:
            correlations.append({
                "type": "EXACT_MATCH",
                "field": f'item_total_{item["item_id"]}',
                "value": item_total
            })

    return {
        "transaction_id": transaction["transaction_id"],
        "anomaly_type": health_result["anomaly_type"],
        "investigation_basis": investigation_basis,
        "suspicious_amount": suspicious_amount,
        "correlations": correlations
    }

Run correlation analysis

In [14]:
correlation_results = []

for transaction, health_result in zip(transactions, health_results):

    result = analyze_amount_correlations(
        transaction,
        health_result
    )

    correlation_results.append(result)

    print("=" * 60)
    print("Transaction:", transaction["transaction_id"])

    if health_result["status"] == "NORMAL":
        print("No anomaly - investigation skipped.")
        continue

    print("Anomaly:", result["anomaly_type"])
    print("Investigation Basis:", result["investigation_basis"])
    print("Suspicious Amount: $", result["suspicious_amount"])

    if result["correlations"]:
        print("\nCorrelations found:")

        for correlation in result["correlations"]:
            print(
                "  ->",
                correlation["field"],
                "= $",
                correlation["value"],
                "|",
                correlation["type"]
            )
    else:
        print("\nNo direct amount correlation found.")

Transaction: TXN001
No anomaly - investigation skipped.
Transaction: TXN002
Anomaly: HIGH_CASHBACK
Investigation Basis: EXCESS_CASHBACK
Suspicious Amount: $ 25.0

Correlations found:
  -> gift_card_amount = $ 25.0 | EXACT_MATCH
  -> gift_card_failure_amount = $ 25.0 | EXACT_MATCH
Transaction: TXN003
Anomaly: NEGATIVE_CASHBACK
Investigation Basis: NEGATIVE_CASHBACK_MAGNITUDE
Suspicious Amount: $ 15.0

No direct amount correlation found.


Add correlation strength

In [15]:
def calculate_evidence_strength(correlation_result):

    correlations = correlation_result.get("correlations", [])

    if len(correlations) >= 2:
        return "STRONG"

    elif len(correlations) == 1:
        return "MODERATE"

    else:
        return "INSUFFICIENT"

In [16]:
for result in correlation_results:

    if "correlations" not in result:
        continue

    strength = calculate_evidence_strength(result)

    result["evidence_strength"] = strength

    print(
        result["transaction_id"],
        "-> Evidence Strength:",
        strength
    )

TXN001 -> Evidence Strength: INSUFFICIENT
TXN002 -> Evidence Strength: STRONG
TXN003 -> Evidence Strength: INSUFFICIENT


Part 4 — Historical Transaction Context Analysis

## Part 4: Historical Transaction Context Analysis

Some rare POS anomalies may depend on transaction state carried across
transaction boundaries.

The Historical Context Analyzer examines previous transactions from the
same lane and checks whether earlier failure-related amounts correlate
with the current cashback anomaly.

This analysis generates investigation evidence only. A detected
correlation does not by itself prove causation.

Add a historical edge-case sequence

In [17]:
historical_transactions = [
    {
        "transaction_id": "TXN004",
        "store_id": "STORE103",
        "lane_id": "LANE05",
        "sequence": 1,

        "items": [
            {"item_id": "ITEM301", "price": 30.00, "quantity": 1}
        ],

        "basket_total": 30.00,

        "promotion": {
            "discount_amount": 0.00
        },

        "gift_card": {
            "used": True,
            "amount": 10.00,
            "activation_status": "FAILED",
            "failure_amount": 10.00
        },

        "payment": {
            "type": "CREDIT",
            "amount": 30.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 0.00,
            "actual": 0.00
        }
    },

    {
        "transaction_id": "TXN005",
        "store_id": "STORE103",
        "lane_id": "LANE05",
        "sequence": 2,

        "items": [
            {"item_id": "ITEM302", "price": 40.00, "quantity": 1}
        ],

        "basket_total": 40.00,

        "promotion": {
            "discount_amount": 0.00
        },

        "gift_card": {
            "used": True,
            "amount": 15.00,
            "activation_status": "FAILED",
            "failure_amount": 15.00
        },

        "payment": {
            "type": "DEBIT",
            "amount": 40.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 0.00,
            "actual": 0.00
        }
    },

    {
        "transaction_id": "TXN006",
        "store_id": "STORE103",
        "lane_id": "LANE05",
        "sequence": 3,

        "items": [
            {"item_id": "ITEM303", "price": 50.00, "quantity": 1}
        ],

        "basket_total": 50.00,

        "promotion": {
            "discount_amount": 5.00
        },

        "gift_card": {
            "used": False,
            "amount": 0.00,
            "activation_status": "NOT_APPLICABLE",
            "failure_amount": 0.00
        },

        "payment": {
            "type": "CREDIT",
            "amount": 45.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": 30.00
        }
    }
]

print("Historical transactions created:", len(historical_transactions))

Historical transactions created: 3


4B — Historical Context Analyzer

In [18]:
def analyze_historical_context(current_transaction, transaction_history):

    current_health = check_transaction_health(current_transaction)

    # Historical analysis is only required for anomalies
    if current_health["status"] == "NORMAL":
        return {
            "transaction_id": current_transaction["transaction_id"],
            "historical_correlation": False,
            "message": "Normal transaction - historical analysis not required."
        }

    # Determine suspicious amount
    if current_health["anomaly_type"] == "HIGH_CASHBACK":
        suspicious_amount = round(
            current_health["actual_cashback"]
            - current_health["expected_cashback"],
            2
        )

    elif current_health["anomaly_type"] == "NEGATIVE_CASHBACK":
        suspicious_amount = round(
            abs(current_health["actual_cashback"]),
            2
        )

    else:
        suspicious_amount = 0.0

    # Only compare previous transactions from same store/lane
    relevant_history = [
        tx for tx in transaction_history
        if tx["store_id"] == current_transaction["store_id"]
        and tx["lane_id"] == current_transaction["lane_id"]
        and tx.get("sequence", 0) < current_transaction.get("sequence", 0)
    ]

    previous_failure_amounts = [
        tx["gift_card"]["failure_amount"]
        for tx in relevant_history
        if tx["gift_card"]["failure_amount"] > 0
    ]

    accumulated_failure_amount = round(
        sum(previous_failure_amounts),
        2
    )

    historical_match = (
        accumulated_failure_amount > 0
        and abs(suspicious_amount - accumulated_failure_amount) < 0.01
    )

    return {
        "transaction_id": current_transaction["transaction_id"],
        "anomaly_type": current_health["anomaly_type"],
        "suspicious_amount": suspicious_amount,
        "previous_failure_amounts": previous_failure_amounts,
        "accumulated_failure_amount": accumulated_failure_amount,
        "historical_correlation": historical_match
    }

Investigate TXN006

In [19]:
current_transaction = historical_transactions[2]

history_result = analyze_historical_context(
    current_transaction,
    historical_transactions
)

print("Transaction:", history_result["transaction_id"])
print("Anomaly Type:", history_result["anomaly_type"])
print("Suspicious Cashback Amount: $", history_result["suspicious_amount"])

print(
    "Previous Gift Card Failure Amounts:",
    history_result["previous_failure_amounts"]
)

print(
    "Accumulated Previous Failure Amount: $",
    history_result["accumulated_failure_amount"]
)

print(
    "Historical Correlation:",
    history_result["historical_correlation"]
)

Transaction: TXN006
Anomaly Type: HIGH_CASHBACK
Suspicious Cashback Amount: $ 25.0
Previous Gift Card Failure Amounts: [10.0, 15.0]
Accumulated Previous Failure Amount: $ 25.0
Historical Correlation: True


Generate structured evidence

In [20]:
def build_historical_evidence(history_result):

    if history_result.get("historical_correlation"):

        return {
            "evidence_type": "CROSS_TRANSACTION_CORRELATION",
            "strength": "STRONG",
            "observation": (
                f"The suspicious cashback amount of "
                f"${history_result['suspicious_amount']:.2f} "
                f"matches the accumulated previous gift-card "
                f"failure amount of "
                f"${history_result['accumulated_failure_amount']:.2f}."
            ),
            "interpretation": (
                "This may indicate that failure-related state "
                "persisted across transaction boundaries. "
                "Further investigation is required."
            )
        }

    return {
        "evidence_type": "NO_HISTORICAL_CORRELATION",
        "strength": "INSUFFICIENT",
        "observation": "No matching historical amount pattern was identified.",
        "interpretation": "No historical root-cause hypothesis can be supported."
    }


historical_evidence = build_historical_evidence(history_result)

print(json.dumps(historical_evidence, indent=2))

{
  "evidence_type": "CROSS_TRANSACTION_CORRELATION",
  "strength": "STRONG",
  "observation": "The suspicious cashback amount of $25.00 matches the accumulated previous gift-card failure amount of $25.00.",
  "interpretation": "This may indicate that failure-related state persisted across transaction boundaries. Further investigation is required."
}


Part 5 — RAG Knowledge Base

## Part 5: RAG-Based POS Knowledge Retrieval

The investigation assistant uses Retrieval-Augmented Generation (RAG)
to retrieve relevant POS business and troubleshooting knowledge.

The knowledge base contains synthetic guidance related to:

- Cashback processing
- Gift card failure handling
- Transaction state management
- Negative cashback investigation
- Escalation procedures

RAG helps ground the investigation in retrieved evidence rather than
allowing the LLM to generate unsupported explanations.

Create the synthetic knowledge base

In [21]:
knowledge_documents = [
    {
        "doc_id": "KB001",
        "title": "POS Cashback Processing Guide",
        "content": """
Cashback values must be validated against the expected cashback
calculated for the current transaction.

If actual cashback is greater than expected cashback, the transaction
must be treated as a high-cashback anomaly.

Investigation should compare the cashback variance with other monetary
values in the transaction, including promotion amounts, tender values,
gift card amounts and failure-related amounts.

An amount correlation is investigation evidence and must not by itself
be considered proof of root cause.
"""
    },

    {
        "doc_id": "KB002",
        "title": "Gift Card Failure Handling Guide",
        "content": """
Gift card processing may contain transaction-scoped values representing
failed or incomplete gift card operations.

Failure-related values should be associated with the transaction in
which the failure occurred.

When investigating abnormal cashback after a gift card failure,
engineers should verify the failure amount, activation status and
subsequent transaction processing.

Failure-related state should not incorrectly influence unrelated
transactions.
"""
    },

    {
        "doc_id": "KB003",
        "title": "POS Transaction State Management Guide",
        "content": """
Transaction-specific temporary state should be initialized correctly
for each new POS transaction.

State associated with a completed or failed transaction should not
unexpectedly affect subsequent transactions.

If an anomalous monetary value matches accumulated values from previous
transactions, investigate whether transaction-scoped state was retained
across transaction boundaries.

A cross-transaction monetary correlation should be treated as a
root-cause hypothesis requiring engineering validation.
"""
    },

    {
        "doc_id": "KB004",
        "title": "Negative Cashback Investigation Guide",
        "content": """
A negative cashback value should be treated as an anomaly and requires
investigation.

The investigation should examine adjustments, reversals, promotions,
refund-related values and tender events that may correlate with the
magnitude of the negative cashback.

If no supporting transaction evidence is available, the system should
not infer a root cause and should recommend manual investigation.
"""
    },

    {
        "doc_id": "KB005",
        "title": "POS Anomaly Escalation Guide",
        "content": """
An automated investigation should distinguish between observations,
correlations and confirmed root causes.

When evidence is insufficient or conflicting, the investigation result
should be marked as inconclusive.

Inconclusive anomalies should be escalated for manual engineering
investigation.

AI-generated hypotheses must remain subject to human review before
being accepted as a root-cause conclusion.
"""
    }
]

print("Knowledge documents created:", len(knowledge_documents))

Knowledge documents created: 5


Install embedding/vector-search libraries

In [22]:
!pip -q install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 39.5 MB/s eta 0:00:00


Load the embedding model

In [23]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


Create document embeddings

In [24]:
document_texts = [
    doc["title"] + "\n" + doc["content"]
    for doc in knowledge_documents
]

document_embeddings = embedding_model.encode(
    document_texts,
    convert_to_numpy=True
)

print("Number of documents:", len(document_texts))
print("Embedding shape:", document_embeddings.shape)

Number of documents: 5
Embedding shape: (5, 384)


Build the FAISS vector database

In [25]:
dimension = document_embeddings.shape[1]

faiss_index = faiss.IndexFlatL2(dimension)

faiss_index.add(
    document_embeddings.astype("float32")
)

print("Documents stored in FAISS:", faiss_index.ntotal)

Documents stored in FAISS: 5


Build the Retriever

In [26]:
def retrieve_pos_knowledge(query, top_k=2):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for distance, index in zip(distances[0], indices[0]):

        document = knowledge_documents[index]

        results.append({
            "doc_id": document["doc_id"],
            "title": document["title"],
            "content": document["content"].strip(),
            "distance": float(distance)
        })

    return results

Test RAG retrieval

In [27]:
query = """
A high cashback transaction has an excess amount that matches
accumulated gift card failure amounts from previous transactions.
What should be investigated?
"""

retrieved_documents = retrieve_pos_knowledge(
    query,
    top_k=2
)

for doc in retrieved_documents:

    print("=" * 70)
    print("Document:", doc["title"])
    print("Distance:", round(doc["distance"], 4))
    print()
    print(doc["content"])

Document: Gift Card Failure Handling Guide
Distance: 0.6626

Gift card processing may contain transaction-scoped values representing
failed or incomplete gift card operations.

Failure-related values should be associated with the transaction in
which the failure occurred.

When investigating abnormal cashback after a gift card failure,
engineers should verify the failure amount, activation status and
subsequent transaction processing.

Failure-related state should not incorrectly influence unrelated
transactions.
Document: POS Cashback Processing Guide
Distance: 0.8364

Cashback values must be validated against the expected cashback
calculated for the current transaction.

If actual cashback is greater than expected cashback, the transaction
must be treated as a high-cashback anomaly.

Investigation should compare the cashback variance with other monetary
values in the transaction, including promotion amounts, tender values,
gift card amounts and failure-related amounts.

An amount cor

Connect the anomaly evidence to RAG

In [28]:
def build_rag_query(health_result, historical_evidence):

    query = f"""
POS transaction anomaly investigation.

Anomaly type:
{health_result['anomaly_type']}

Expected cashback:
${health_result['expected_cashback']:.2f}

Actual cashback:
${health_result['actual_cashback']:.2f}

Cashback variance:
${health_result['variance']:.2f}

Historical evidence:
{historical_evidence['observation']}

Evidence interpretation:
{historical_evidence['interpretation']}

Retrieve POS knowledge that can help investigate this anomaly.
"""

    return query

In [29]:
txn006_health = check_transaction_health(
    historical_transactions[2]
)

rag_query = build_rag_query(
    txn006_health,
    historical_evidence
)

print(rag_query)


POS transaction anomaly investigation.

Anomaly type:
HIGH_CASHBACK

Expected cashback:
$5.00

Actual cashback:
$30.00

Cashback variance:
$25.00

Historical evidence:
The suspicious cashback amount of $25.00 matches the accumulated previous gift-card failure amount of $25.00.

Evidence interpretation:
This may indicate that failure-related state persisted across transaction boundaries. Further investigation is required.

Retrieve POS knowledge that can help investigate this anomaly.



In [30]:
txn006_knowledge = retrieve_pos_knowledge(
    rag_query,
    top_k=2
)

for doc in txn006_knowledge:

    print("=" * 70)
    print(doc["title"])
    print("=" * 70)
    print(doc["content"])

POS Cashback Processing Guide
Cashback values must be validated against the expected cashback
calculated for the current transaction.

If actual cashback is greater than expected cashback, the transaction
must be treated as a high-cashback anomaly.

Investigation should compare the cashback variance with other monetary
values in the transaction, including promotion amounts, tender values,
gift card amounts and failure-related amounts.

An amount correlation is investigation evidence and must not by itself
be considered proof of root cause.
POS Transaction State Management Guide
Transaction-specific temporary state should be initialized correctly
for each new POS transaction.

State associated with a completed or failed transaction should not
unexpectedly affect subsequent transactions.

If an anomalous monetary value matches accumulated values from previous
transactions, investigate whether transaction-scoped state was retained
across transaction boundaries.

A cross-transaction moneta

Part 6 — GenAI Investigation Agent

## Part 6: GenAI Investigation Agent

The GenAI Investigation Agent combines:

- Transaction details
- Deterministic health-check results
- Current transaction correlations
- Historical transaction evidence
- Retrieved RAG knowledge

The agent generates an evidence-grounded investigation hypothesis and
recommended next steps.

The LLM does not perform the financial calculations and must not claim
a confirmed root cause when supporting evidence is insufficient.

Install Gemini SDK

In [31]:
!pip -q install google-genai

In [32]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [33]:
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY").strip()

print("Key loaded:", bool(api_key))
print("Contains newline:", "\n" in api_key)
print("Starts with AQ:", api_key.startswith("AQ"))

Key loaded: True
Contains newline: False
Starts with AQ: True


In [34]:
from google import genai

client = genai.Client(api_key=api_key)

print("Client created")

Client created


In [35]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Reply with only: GEMINI CONNECTION SUCCESSFUL"
)

print(response.text)

GEMINI CONNECTION SUCCESSFUL


Part 7 — Synthetic POS Transaction Dataset

## Part 7: Synthetic POS Transaction Dataset

For this capstone prototype, synthetic retail POS transactions are used
to demonstrate cashback anomaly investigation.

Each transaction contains information such as:

- Store and lane
- Basket amount
- Promotion amount
- Gift card processing status
- Gift card failure amount
- Payment information
- Expected cashback
- Actual cashback

The dataset includes normal transactions as well as high and negative
cashback scenarios.

Synthetic data is used so that no production or customer information
is exposed.

In [42]:
transactions = [

    # ---------------------------------------------------------
    # TXN001 - Normal transaction
    # ---------------------------------------------------------
    {
        "transaction_id": "TXN001",
        "store_id": "STORE101",
        "lane_id": "LANE01",
        "sequence": 1,

        "items": [
            {
                "item_id": "ITEM101",
                "price": 20.00,
                "quantity": 2
            }
        ],

        "basket_total": 40.00,

        "promotion": {
            "discount_amount": 0.00
        },

        "gift_card": {
            "used": False,
            "amount": 0.00,
            "activation_status": "NOT_APPLICABLE",
            "failure_amount": 0.00
        },

        "payment": {
            "type": "DEBIT",
            "amount": 40.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": 5.00
        }
    },


    # ---------------------------------------------------------
    # TXN002 - High cashback with CURRENT transaction evidence
    # ---------------------------------------------------------
    {
        "transaction_id": "TXN002",
        "store_id": "STORE101",
        "lane_id": "LANE02",
        "sequence": 1,

        "items": [
            {
                "item_id": "ITEM201",
                "price": 50.00,
                "quantity": 1
            }
        ],

        "basket_total": 50.00,

        "promotion": {
            "discount_amount": 0.00
        },

        "gift_card": {
            "used": True,
            "amount": 20.00,
            "activation_status": "FAILED",
            "failure_amount": 20.00
        },

        "payment": {
            "type": "DEBIT",
            "amount": 50.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": 25.00
        }
    },


    # ---------------------------------------------------------
    # TXN003 - Negative cashback
    # ---------------------------------------------------------
    {
        "transaction_id": "TXN003",
        "store_id": "STORE102",
        "lane_id": "LANE01",
        "sequence": 1,

        "items": [
            {
                "item_id": "ITEM301",
                "price": 30.00,
                "quantity": 1
            }
        ],

        "basket_total": 30.00,

        "promotion": {
            "discount_amount": 0.00
        },

        "gift_card": {
            "used": False,
            "amount": 0.00,
            "activation_status": "NOT_APPLICABLE",
            "failure_amount": 0.00
        },

        "payment": {
            "type": "CREDIT",
            "amount": 30.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": -15.00
        }
    }
]

In [43]:
historical_transactions = [

    # Previous transaction - Gift card failure $10
    {
        "transaction_id": "TXN004",
        "store_id": "STORE103",
        "lane_id": "LANE03",
        "sequence": 1,

        "items": [
            {
                "item_id": "ITEM501",
                "price": 40.00,
                "quantity": 1
            }
        ],

        "basket_total": 40.00,

        "promotion": {
            "discount_amount": 0.00
        },

        "gift_card": {
            "used": True,
            "amount": 10.00,
            "activation_status": "FAILED",
            "failure_amount": 10.00
        },

        "payment": {
            "type": "DEBIT",
            "amount": 40.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": 5.00
        }
    },


    # Previous transaction - Gift card failure $15
    {
        "transaction_id": "TXN005",
        "store_id": "STORE103",
        "lane_id": "LANE03",
        "sequence": 2,

        "items": [
            {
                "item_id": "ITEM502",
                "price": 35.00,
                "quantity": 1
            }
        ],

        "basket_total": 35.00,

        "promotion": {
            "discount_amount": 0.00
        },

        "gift_card": {
            "used": True,
            "amount": 15.00,
            "activation_status": "FAILED",
            "failure_amount": 15.00
        },

        "payment": {
            "type": "DEBIT",
            "amount": 35.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": 5.00
        }
    },


    # Current anomalous transaction
    {
        "transaction_id": "TXN006",
        "store_id": "STORE103",
        "lane_id": "LANE03",
        "sequence": 3,

        "items": [
            {
                "item_id": "ITEM503",
                "price": 60.00,
                "quantity": 1
            }
        ],

        "basket_total": 60.00,

        "promotion": {
            "discount_amount": 0.00
        },

        "gift_card": {
            "used": False,
            "amount": 0.00,
            "activation_status": "NOT_APPLICABLE",
            "failure_amount": 0.00
        },

        "payment": {
            "type": "DEBIT",
            "amount": 60.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": 30.00
        }
    }
]

In [44]:
all_transactions = (
    transactions +
    historical_transactions
)

print(
    "Total transactions:",
    len(all_transactions)
)

print()

for txn in all_transactions:

    expected = txn["cashback"]["expected"]
    actual = txn["cashback"]["actual"]

    print(
        txn["transaction_id"],
        "| Store:", txn["store_id"],
        "| Lane:", txn["lane_id"],
        "| Expected Cashback:", expected,
        "| Actual Cashback:", actual,
        "| Variance:", round(actual - expected, 2)
    )

Total transactions: 6

TXN001 | Store: STORE101 | Lane: LANE01 | Expected Cashback: 5.0 | Actual Cashback: 5.0 | Variance: 0.0
TXN002 | Store: STORE101 | Lane: LANE02 | Expected Cashback: 5.0 | Actual Cashback: 25.0 | Variance: 20.0
TXN003 | Store: STORE102 | Lane: LANE01 | Expected Cashback: 5.0 | Actual Cashback: -15.0 | Variance: -20.0
TXN004 | Store: STORE103 | Lane: LANE03 | Expected Cashback: 5.0 | Actual Cashback: 5.0 | Variance: 0.0
TXN005 | Store: STORE103 | Lane: LANE03 | Expected Cashback: 5.0 | Actual Cashback: 5.0 | Variance: 0.0
TXN006 | Store: STORE103 | Lane: LANE03 | Expected Cashback: 5.0 | Actual Cashback: 30.0 | Variance: 25.0


Part 8 — Transaction Health & Anomaly Detection

## Part 8: Transaction Health & Anomaly Detection Agent

The Transaction Health Agent performs deterministic validation of
cashback values.

It compares expected cashback with actual cashback and calculates
the variance.

The transaction is classified as:

- NORMAL
- HIGH_CASHBACK
- NEGATIVE_CASHBACK

Financial calculations are performed using deterministic Python logic
rather than an LLM.

Create the health-check function

In [45]:
def check_transaction_health(transaction):

    expected = float(
        transaction["cashback"]["expected"]
    )

    actual = float(
        transaction["cashback"]["actual"]
    )

    variance = round(
        actual - expected,
        2
    )

    # Determine anomaly
    if actual < 0:

        anomaly_type = "NEGATIVE_CASHBACK"
        status = "ANOMALY"

    elif variance > 0:

        anomaly_type = "HIGH_CASHBACK"
        status = "ANOMALY"

    else:

        anomaly_type = "NONE"
        status = "NORMAL"

    return {
        "transaction_id":
            transaction["transaction_id"],

        "status":
            status,

        "anomaly_type":
            anomaly_type,

        "expected_cashback":
            expected,

        "actual_cashback":
            actual,

        "variance":
            variance
    }

In [46]:
health_results = []

for txn in all_transactions:

    result = check_transaction_health(txn)

    health_results.append(result)

    print(
        result["transaction_id"],
        "| Status:", result["status"],
        "| Anomaly:", result["anomaly_type"],
        "| Variance:", result["variance"]
    )

TXN001 | Status: NORMAL | Anomaly: NONE | Variance: 0.0
TXN002 | Status: ANOMALY | Anomaly: HIGH_CASHBACK | Variance: 20.0
TXN003 | Status: ANOMALY | Anomaly: NEGATIVE_CASHBACK | Variance: -20.0
TXN004 | Status: NORMAL | Anomaly: NONE | Variance: 0.0
TXN005 | Status: NORMAL | Anomaly: NONE | Variance: 0.0
TXN006 | Status: ANOMALY | Anomaly: HIGH_CASHBACK | Variance: 25.0


Create the Router

In [47]:
def route_transaction(health_result):

    if health_result["status"] == "NORMAL":
        return "NORMAL_PATH"

    elif health_result["anomaly_type"] == "HIGH_CASHBACK":
        return "HIGH_CASHBACK_PATH"

    elif health_result["anomaly_type"] == "NEGATIVE_CASHBACK":
        return "NEGATIVE_CASHBACK_PATH"

    else:
        return "MANUAL_REVIEW_PATH"

In [48]:
for result in health_results:

    route = route_transaction(result)

    print(
        result["transaction_id"],
        "->",
        route
    )

TXN001 -> NORMAL_PATH
TXN002 -> HIGH_CASHBACK_PATH
TXN003 -> NEGATIVE_CASHBACK_PATH
TXN004 -> NORMAL_PATH
TXN005 -> NORMAL_PATH
TXN006 -> HIGH_CASHBACK_PATH


Make the output more presentable

In [49]:
import pandas as pd

health_df = pd.DataFrame(
    health_results
)

display(health_df)

,transaction_id,status,anomaly_type,expected_cashback,actual_cashback,variance
0,TXN001,NORMAL,NONE,5.0,5.0,0.0
1,TXN002,ANOMALY,HIGH_CASHBACK,5.0,25.0,20.0
2,TXN003,ANOMALY,NEGATIVE_CASHBACK,5.0,-15.0,-20.0
3,TXN004,NORMAL,NONE,5.0,5.0,0.0
4,TXN005,NORMAL,NONE,5.0,5.0,0.0
5,TXN006,ANOMALY,HIGH_CASHBACK,5.0,30.0,25.0


Part 9 — Pattern Analysis Agent

## Part 9: Current Transaction Pattern Analysis Agent

The Pattern Analysis Agent investigates anomalous cashback by comparing
the cashback variance with monetary values available within the current
POS transaction.

The agent checks potential correlations with:

- Gift card failure amount
- Gift card amount
- Promotion discount amount
- Payment amount
- Basket amount

A matching amount is treated only as investigation evidence and not as
proof of root cause.

Create the Pattern Analysis Agent

In [50]:
def pattern_analysis_agent(transaction, health_result):

    variance = round(
        abs(health_result["variance"]),
        2
    )

    correlations = []

    # --------------------------------------------------
    # Gift card failure amount
    # --------------------------------------------------

    giftcard_failure = round(
        float(
            transaction["gift_card"]["failure_amount"]
        ),
        2
    )

    if giftcard_failure > 0 and giftcard_failure == variance:

        correlations.append({
            "field": "gift_card_failure_amount",
            "value": giftcard_failure,
            "relationship":
                "Matches absolute cashback variance"
        })

    # --------------------------------------------------
    # Gift card amount
    # --------------------------------------------------

    giftcard_amount = round(
        float(
            transaction["gift_card"]["amount"]
        ),
        2
    )

    if (
        giftcard_amount > 0
        and giftcard_amount == variance
    ):

        correlations.append({
            "field": "gift_card_amount",
            "value": giftcard_amount,
            "relationship":
                "Matches absolute cashback variance"
        })

    # --------------------------------------------------
    # Promotion amount
    # --------------------------------------------------

    promotion_amount = round(
        float(
            transaction["promotion"]["discount_amount"]
        ),
        2
    )

    if (
        promotion_amount > 0
        and promotion_amount == variance
    ):

        correlations.append({
            "field": "promotion_discount_amount",
            "value": promotion_amount,
            "relationship":
                "Matches absolute cashback variance"
        })

    # --------------------------------------------------
    # Payment amount
    # --------------------------------------------------

    payment_amount = round(
        float(
            transaction["payment"]["amount"]
        ),
        2
    )

    if (
        payment_amount > 0
        and payment_amount == variance
    ):

        correlations.append({
            "field": "payment_amount",
            "value": payment_amount,
            "relationship":
                "Matches absolute cashback variance"
        })

    # --------------------------------------------------
    # Basket amount
    # --------------------------------------------------

    basket_total = round(
        float(transaction["basket_total"]),
        2
    )

    if (
        basket_total > 0
        and basket_total == variance
    ):

        correlations.append({
            "field": "basket_total",
            "value": basket_total,
            "relationship":
                "Matches absolute cashback variance"
        })

    return {
        "transaction_id":
            transaction["transaction_id"],

        "cashback_variance":
            health_result["variance"],

        "absolute_variance":
            variance,

        "correlations":
            correlations,

        "correlation_found":
            len(correlations) > 0
    }

Test TXN002

In [51]:
txn002 = transactions[1]

txn002_health = check_transaction_health(
    txn002
)

txn002_pattern = pattern_analysis_agent(
    txn002,
    txn002_health
)

print(txn002_pattern)

{'transaction_id': 'TXN002', 'cashback_variance': 20.0, 'absolute_variance': 20.0, 'correlations': [{'field': 'gift_card_failure_amount', 'value': 20.0, 'relationship': 'Matches absolute cashback variance'}, {'field': 'gift_card_amount', 'value': 20.0, 'relationship': 'Matches absolute cashback variance'}], 'correlation_found': True}


Test TXN006

In [52]:
txn006 = historical_transactions[2]

txn006_health = check_transaction_health(
    txn006
)

txn006_pattern = pattern_analysis_agent(
    txn006,
    txn006_health
)

print(txn006_pattern)

{'transaction_id': 'TXN006', 'cashback_variance': 25.0, 'absolute_variance': 25.0, 'correlations': [], 'correlation_found': False}


Test TXN003

In [53]:
txn003 = transactions[2]

txn003_health = check_transaction_health(
    txn003
)

txn003_pattern = pattern_analysis_agent(
    txn003,
    txn003_health
)

print(txn003_pattern)

{'transaction_id': 'TXN003', 'cashback_variance': -20.0, 'absolute_variance': 20.0, 'correlations': [], 'correlation_found': False}


Display a clean summary

In [54]:
pattern_summary = []

for txn in [
    transactions[1],          # TXN002
    transactions[2],          # TXN003
    historical_transactions[2] # TXN006
]:

    health = check_transaction_health(txn)

    pattern = pattern_analysis_agent(
        txn,
        health
    )

    matched_fields = [
        item["field"]
        for item in pattern["correlations"]
    ]

    pattern_summary.append({
        "Transaction":
            txn["transaction_id"],

        "Anomaly":
            health["anomaly_type"],

        "Variance":
            health["variance"],

        "Current Correlation":
            pattern["correlation_found"],

        "Matched Fields":
            ", ".join(matched_fields)
            if matched_fields
            else "None"
    })

In [55]:
pattern_df = pd.DataFrame(
    pattern_summary
)

display(pattern_df)

,Transaction,Anomaly,Variance,Current Correlation,Matched Fields
0,TXN002,HIGH_CASHBACK,20.0,True,"gift_card_failure_amount, gift_card_amount"
1,TXN003,NEGATIVE_CASHBACK,-20.0,False,None
2,TXN006,HIGH_CASHBACK,25.0,False,None


Part 10 — Historical Context Agent

## Part 10: Historical Context Agent

The Historical Context Agent investigates transactions that cannot be
fully explained using the current transaction alone.

It searches earlier transactions from the same store and lane and
examines relevant historical events such as gift-card activation
failures.

The agent compares the accumulated historical failure amount with the
current cashback anomaly.

A matching value is treated as correlation evidence only and does not
confirm causation.

Create the Historical Context Agent

In [56]:
def historical_context_agent(
    current_transaction,
    transaction_history
):

    current_id = current_transaction["transaction_id"]
    current_store = current_transaction["store_id"]
    current_lane = current_transaction["lane_id"]
    current_sequence = current_transaction["sequence"]

    health = check_transaction_health(
        current_transaction
    )

    anomaly_amount = round(
        abs(health["variance"]),
        2
    )

    # Find earlier transactions on same store/lane
    previous_transactions = [
        txn
        for txn in transaction_history

        if txn["store_id"] == current_store
        and txn["lane_id"] == current_lane
        and txn["sequence"] < current_sequence
    ]

    # Sort in transaction sequence order
    previous_transactions = sorted(
        previous_transactions,
        key=lambda x: x["sequence"]
    )

    failure_events = []

    total_failure_amount = 0.0

    for txn in previous_transactions:

        gift_card = txn["gift_card"]

        failure_amount = float(
            gift_card["failure_amount"]
        )

        if (
            gift_card["activation_status"] == "FAILED"
            and failure_amount > 0
        ):

            total_failure_amount += failure_amount

            failure_events.append({
                "transaction_id":
                    txn["transaction_id"],

                "sequence":
                    txn["sequence"],

                "failure_amount":
                    failure_amount
            })

    total_failure_amount = round(
        total_failure_amount,
        2
    )

    historical_correlation = (
        total_failure_amount > 0
        and total_failure_amount == anomaly_amount
    )

    return {
        "transaction_id":
            current_id,

        "anomaly_amount":
            anomaly_amount,

        "previous_transactions_checked":
            len(previous_transactions),

        "gift_card_failure_events":
            failure_events,

        "accumulated_failure_amount":
            total_failure_amount,

        "analysis": {
            "historical_correlation":
                historical_correlation
        },

        "evidence": {
            "observation":
                (
                    f"Current cashback anomaly amount "
                    f"is ${anomaly_amount:.2f}. "
                    f"Accumulated previous gift-card "
                    f"failure amount is "
                    f"${total_failure_amount:.2f}."
                )
        }
    }

Test our main TXN006 scenario

In [57]:
txn006_history = historical_context_agent(
    historical_transactions[2],
    historical_transactions
)

print(txn006_history)

{'transaction_id': 'TXN006', 'anomaly_amount': 25.0, 'previous_transactions_checked': 2, 'gift_card_failure_events': [{'transaction_id': 'TXN004', 'sequence': 1, 'failure_amount': 10.0}, {'transaction_id': 'TXN005', 'sequence': 2, 'failure_amount': 15.0}], 'accumulated_failure_amount': 25.0, 'analysis': {'historical_correlation': True}, 'evidence': {'observation': 'Current cashback anomaly amount is $25.00. Accumulated previous gift-card failure amount is $25.00.'}}


Print it in a presentation-friendly format

In [58]:
print("=" * 65)
print("HISTORICAL CONTEXT INVESTIGATION")
print("=" * 65)

print(
    "Transaction:",
    txn006_history["transaction_id"]
)

print(
    "Cashback anomaly amount: $",
    txn006_history["anomaly_amount"]
)

print()

print("Previous gift-card failures:")

for event in txn006_history[
    "gift_card_failure_events"
]:

    print(
        " ",
        event["transaction_id"],
        "-> $",
        event["failure_amount"]
    )

print()

print(
    "Accumulated failure amount: $",
    txn006_history[
        "accumulated_failure_amount"
    ]
)

print(
    "Historical correlation:",
    txn006_history[
        "analysis"
    ]["historical_correlation"]
)

HISTORICAL CONTEXT INVESTIGATION
Transaction: TXN006
Cashback anomaly amount: $ 25.0

Previous gift-card failures:
  TXN004 -> $ 10.0
  TXN005 -> $ 15.0

Accumulated failure amount: $ 25.0
Historical correlation: True


Test that the agent doesn't falsely correlate TXN003

In [59]:
txn003_history = historical_context_agent(
    transactions[2],
    all_transactions
)

print(
    "Transaction:",
    txn003_history["transaction_id"]
)

print(
    "Anomaly amount:",
    txn003_history["anomaly_amount"]
)

print(
    "Accumulated historical failure:",
    txn003_history[
        "accumulated_failure_amount"
    ]
)

print(
    "Historical correlation:",
    txn003_history[
        "analysis"
    ]["historical_correlation"]
)

Transaction: TXN003
Anomaly amount: 20.0
Accumulated historical failure: 0.0
Historical correlation: False


Combine Pattern + Historical Evidence

In [60]:
def investigate_evidence(
    transaction,
    transaction_history
):

    health = check_transaction_health(
        transaction
    )

    pattern = pattern_analysis_agent(
        transaction,
        health
    )

    history = historical_context_agent(
        transaction,
        transaction_history
    )

    if pattern["correlation_found"]:

        evidence_status = (
            "CURRENT_TRANSACTION_CORRELATION"
        )

    elif (
        history["analysis"][
            "historical_correlation"
        ]
    ):

        evidence_status = (
            "HISTORICAL_CORRELATION"
        )

    else:

        evidence_status = (
            "INSUFFICIENT_EVIDENCE"
        )

    return {
        "transaction":
            transaction,

        "health_result":
            health,

        "pattern_evidence":
            pattern,

        "historical_evidence":
            history,

        "evidence_status":
            evidence_status
    }

Test the three important cases

In [61]:
for txn in [
    transactions[1],           # TXN002
    transactions[2],           # TXN003
    historical_transactions[2] # TXN006
]:

    result = investigate_evidence(
        txn,
        all_transactions
    )

    print(
        txn["transaction_id"],
        "->",
        result["evidence_status"]
    )

TXN002 -> CURRENT_TRANSACTION_CORRELATION
TXN003 -> INSUFFICIENT_EVIDENCE
TXN006 -> HISTORICAL_CORRELATION


Part 11 — POS Knowledge Base + FAISS RAG

Step 11A — Install packages

In [62]:
!pip -q install sentence-transformers faiss-cpu

In [63]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("RAG libraries loaded successfully")

RAG libraries loaded successfully


Create our prototype POS knowledge base

## Part 11: Retrieval-Augmented Generation (RAG)

The RAG layer provides relevant POS troubleshooting knowledge to the
GenAI RCA Agent.

For the prototype, a synthetic POS knowledge base is created covering:

- Cashback anomaly investigation
- Gift-card activation failures
- Transaction-state management
- Negative cashback investigation
- Human escalation

The documents are converted into embeddings and stored in a FAISS
vector index.

During investigation, relevant knowledge is retrieved based on the
transaction evidence and supplied to Gemini along with the factual
evidence discovered by the deterministic agents.

In [64]:
pos_knowledge_base = [

    {
        "title": "Cashback Anomaly Investigation Guide",

        "content": """
When actual cashback differs from expected cashback, calculate the
cashback variance first.

Investigate whether the variance corresponds to another monetary value
in the transaction such as gift-card amount, failed tender amount,
promotion amount, payment amount or other transaction totals.

A matching amount should be treated as correlation evidence and not
as confirmation of root cause.
"""
    },

    {
        "title": "Gift Card Failure Handling Guide",

        "content": """
Gift-card activation failures should be handled without incorrectly
affecting cashback or subsequent transaction processing.

Failure-related values should be reviewed to ensure that they are
properly initialized, updated and cleared according to the intended
transaction lifecycle.

When investigating a cashback anomaly after gift-card failure,
engineers should review failure handling, transaction cleanup and
relevant POS logs.
"""
    },

    {
        "title": "POS Transaction State Management Guide",

        "content": """
Transaction-specific state should not unintentionally affect later
transactions.

When a suspicious amount in a current transaction matches values from
previous transactions on the same lane, engineers should investigate
whether transaction-specific state was correctly reset or initialized
at transaction boundaries.

Historical monetary correlation alone does not prove that state
persistence caused the issue.
"""
    },

    {
        "title": "Negative Cashback Investigation Guide",

        "content": """
Negative cashback should be treated as an anomalous financial result.

Investigation should validate expected and actual cashback, transaction
type, tender information, promotion information and relevant processing
logs.

If no supporting evidence explains the negative value, the root cause
should remain inconclusive and be escalated for engineering review.
"""
    },

    {
        "title": "POS Investigation Escalation Guide",

        "content": """
An automated investigation should distinguish facts, correlations and
hypotheses.

If transaction evidence is insufficient to establish a meaningful
correlation, the system should not manufacture a root cause.

The case should be marked inconclusive and escalated to an engineer
with the available transaction evidence and relevant logs.
"""
    }
]

print(
    "Knowledge documents:",
    len(pos_knowledge_base)
)

Knowledge documents: 5


Load the embedding model

In [65]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully


Generate embeddings

In [66]:
knowledge_texts = [

    doc["title"] + "\n" + doc["content"]

    for doc in pos_knowledge_base
]

knowledge_embeddings = embedding_model.encode(
    knowledge_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    "Embedding shape:",
    knowledge_embeddings.shape
)

Embedding shape: (5, 384)


Create the FAISS vector database

In [67]:
embedding_dimension = (
    knowledge_embeddings.shape[1]
)

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(
    knowledge_embeddings.astype("float32")
)

print(
    "Documents stored in FAISS:",
    faiss_index.ntotal
)

Documents stored in FAISS: 5


Create the Retriever

In [68]:
def retrieve_pos_knowledge(
    query,
    top_k=2
):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = faiss_index.search(
        query_embedding.astype("float32"),
        top_k
    )

    retrieved_docs = []

    for score, index in zip(
        scores[0],
        indices[0]
    ):

        doc = pos_knowledge_base[index]

        retrieved_docs.append({
            "title":
                doc["title"],

            "content":
                doc["content"],

            "similarity_score":
                float(score)
        })

    return retrieved_docs

Test RAG with TXN006

First create a query from our actual investigation evidence:

In [69]:
txn006_evidence = investigate_evidence(
    historical_transactions[2],
    all_transactions
)

txn006_query = f"""
Investigate a POS {txn006_evidence['health_result']['anomaly_type']}
anomaly.

Cashback variance:
${txn006_evidence['health_result']['variance']}

Current transaction correlation:
{txn006_evidence['pattern_evidence']['correlation_found']}

Historical correlation:
{txn006_evidence['historical_evidence']['analysis']['historical_correlation']}

Accumulated previous gift-card failure amount:
${txn006_evidence['historical_evidence']['accumulated_failure_amount']}
"""

In [70]:
txn006_rag_docs = retrieve_pos_knowledge(
    txn006_query,
    top_k=2
)

for doc in txn006_rag_docs:

    print("=" * 60)

    print(
        "TITLE:",
        doc["title"]
    )

    print(
        "SIMILARITY:",
        round(
            doc["similarity_score"],
            3
        )
    )

    print()

    print(doc["content"])

TITLE: Cashback Anomaly Investigation Guide
SIMILARITY: 0.728


When actual cashback differs from expected cashback, calculate the
cashback variance first.

Investigate whether the variance corresponds to another monetary value
in the transaction such as gift-card amount, failed tender amount,
promotion amount, payment amount or other transaction totals.

A matching amount should be treated as correlation evidence and not
as confirmation of root cause.

TITLE: Gift Card Failure Handling Guide
SIMILARITY: 0.662


Gift-card activation failures should be handled without incorrectly
affecting cashback or subsequent transaction processing.

Failure-related values should be reviewed to ensure that they are
properly initialized, updated and cleared according to the intended
transaction lifecycle.

When investigating a cashback anomaly after gift-card failure,
engineers should review failure handling, transaction cleanup and
relevant POS logs.



Test the negative cashback case

In [71]:
txn003_evidence = investigate_evidence(
    transactions[2],
    all_transactions
)

txn003_query = f"""
Investigate a POS negative cashback anomaly.

Anomaly type:
{txn003_evidence['health_result']['anomaly_type']}

Cashback variance:
${txn003_evidence['health_result']['variance']}

Current correlation:
{txn003_evidence['pattern_evidence']['correlation_found']}

Historical correlation:
{txn003_evidence['historical_evidence']['analysis']['historical_correlation']}

No supporting monetary correlation was identified.
"""

In [72]:
txn003_rag_docs = retrieve_pos_knowledge(
    txn003_query,
    top_k=2
)

for doc in txn003_rag_docs:

    print(
        doc["title"],
        "->",
        round(
            doc["similarity_score"],
            3
        )
    )

Cashback Anomaly Investigation Guide -> 0.667
Negative Cashback Investigation Guide -> 0.637


Create one reusable RAG Agent

In [73]:
def rag_agent(
    investigation_result,
    top_k=2
):

    health = investigation_result[
        "health_result"
    ]

    pattern = investigation_result[
        "pattern_evidence"
    ]

    history = investigation_result[
        "historical_evidence"
    ]

    query = f"""
POS transaction anomaly investigation.

Anomaly type:
{health['anomaly_type']}

Expected cashback:
${health['expected_cashback']}

Actual cashback:
${health['actual_cashback']}

Cashback variance:
${health['variance']}

Current transaction correlation:
{pattern['correlation_found']}

Current correlations:
{pattern['correlations']}

Historical correlation:
{history['analysis']['historical_correlation']}

Historical gift-card failures:
{history['gift_card_failure_events']}

Accumulated historical failure amount:
${history['accumulated_failure_amount']}
"""

    retrieved_docs = retrieve_pos_knowledge(
        query,
        top_k=top_k
    )

    return {
        "query":
            query,

        "retrieved_documents":
            retrieved_docs
    }

In [74]:
rag_result = rag_agent(
    txn006_evidence
)

print("Retrieved POS knowledge:")

for doc in rag_result[
    "retrieved_documents"
]:

    print(
        "-",
        doc["title"]
    )

Retrieved POS knowledge:
- Cashback Anomaly Investigation Guide
- POS Transaction State Management Guide


Part 12 — Gemini GenAI RCA Agent

## Part 12: Gemini GenAI Root Cause Investigation Agent

The Gemini RCA Agent acts as the reasoning and explanation layer of the
POS Transaction Anomaly Investigation Assistant.

The agent receives:

- Deterministically calculated transaction anomaly information
- Current-transaction correlation evidence
- Historical transaction evidence
- POS knowledge retrieved through RAG

Gemini does not perform the financial calculations. Instead, it uses
the grounded evidence to generate an explainable root-cause hypothesis
and recommended engineering investigation steps.

Correlation is not treated as confirmed causation, and all generated
RCA results require human engineering review.

Build the context for Gemini

In [75]:
def build_gemini_context(
    investigation_result,
    rag_result
):

    health = investigation_result[
        "health_result"
    ]

    pattern = investigation_result[
        "pattern_evidence"
    ]

    history = investigation_result[
        "historical_evidence"
    ]

    rag_docs = rag_result[
        "retrieved_documents"
    ]

    knowledge_text = "\n\n".join([
        f"""
DOCUMENT: {doc['title']}
{doc['content']}
"""
        for doc in rag_docs
    ])

    context = f"""
TRANSACTION FACTS
=================

Transaction ID:
{investigation_result['transaction']['transaction_id']}

Store:
{investigation_result['transaction']['store_id']}

Lane:
{investigation_result['transaction']['lane_id']}

Anomaly Type:
{health['anomaly_type']}

Expected Cashback:
${health['expected_cashback']:.2f}

Actual Cashback:
${health['actual_cashback']:.2f}

Cashback Variance:
${health['variance']:.2f}


CURRENT TRANSACTION ANALYSIS
============================

Correlation Found:
{pattern['correlation_found']}

Matched Values:
{pattern['correlations']}


HISTORICAL ANALYSIS
===================

Historical Correlation:
{history['analysis']['historical_correlation']}

Previous Gift Card Failure Events:
{history['gift_card_failure_events']}

Accumulated Historical Failure Amount:
${history['accumulated_failure_amount']:.2f}


EVIDENCE STATUS
===============

{investigation_result['evidence_status']}


RETRIEVED POS KNOWLEDGE
=======================

{knowledge_text}
"""

    return context

Check exactly what we're sending to Gemini

In [76]:
txn006_context = build_gemini_context(
    txn006_evidence,
    rag_result
)

print(txn006_context)


TRANSACTION FACTS

Transaction ID:
TXN006

Store:
STORE103

Lane:
LANE03

Anomaly Type:
HIGH_CASHBACK

Expected Cashback:
$5.00

Actual Cashback:
$30.00

Cashback Variance:
$25.00


CURRENT TRANSACTION ANALYSIS

Correlation Found:
False

Matched Values:
[]


HISTORICAL ANALYSIS

Historical Correlation:
True

Previous Gift Card Failure Events:
[{'transaction_id': 'TXN004', 'sequence': 1, 'failure_amount': 10.0}, {'transaction_id': 'TXN005', 'sequence': 2, 'failure_amount': 15.0}]

Accumulated Historical Failure Amount:
$25.00


EVIDENCE STATUS

HISTORICAL_CORRELATION


RETRIEVED POS KNOWLEDGE


DOCUMENT: Cashback Anomaly Investigation Guide

When actual cashback differs from expected cashback, calculate the
cashback variance first.

Investigate whether the variance corresponds to another monetary value
in the transaction such as gift-card amount, failed tender amount,
promotion amount, payment amount or other transaction totals.

A matching amount should be treated as correlation evide

Create the Gemini RCA Agent

In [79]:
def gemini_rca_agent(
    investigation_result,
    rag_result
):

    context = build_gemini_context(
        investigation_result,
        rag_result
    )

    prompt = f"""
You are an AI assistant supporting engineers who investigate
Retail POS transaction anomalies.

Your task is to analyze the supplied transaction evidence and
retrieved POS knowledge.

IMPORTANT RULES
===============

1. Use only the supplied evidence and retrieved knowledge.
2. Do not invent transaction facts.
3. Financial calculations have already been performed by deterministic
   tools. Do not replace those calculations.
4. Clearly distinguish FACTS, CORRELATIONS and HYPOTHESES.
5. A monetary correlation does not prove causation.
6. Never claim that a probable root cause is confirmed.
7. If evidence is insufficient, state INCONCLUSIVE.
8. Human engineering review is mandatory.

INVESTIGATION CONTEXT
=====================

{context}

Generate the response using exactly these sections:

OBSERVATION:
Summarize the detected anomaly and important monetary values.

SUPPORTING EVIDENCE:
Explain current-transaction and historical evidence.
Explicitly mention matching monetary amounts where relevant.

ROOT-CAUSE HYPOTHESIS:
Provide a cautious technical hypothesis supported by the evidence.
If evidence is insufficient, state INCONCLUSIVE.
Do not claim a hypothesis is confirmed.

RECOMMENDED INVESTIGATION:
List the specific areas an engineer should inspect, including relevant
transaction-state handling, gift-card failure processing, cashback
processing and POS logs where supported by the evidence.

HUMAN REVIEW:
State that human engineering review is required before accepting the
hypothesis as root cause.
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return {
        "transaction_id":
            investigation_result[
                "transaction"
            ]["transaction_id"],

        "model":
            "gemini-3.6-flash",

        "evidence_status":
            investigation_result[
                "evidence_status"
            ],

        "generated_analysis":
            response.text,

        "requires_human_review":
            True
    }

Run Gemini on TXN006

In [80]:
txn006_gemini_result = gemini_rca_agent(
    txn006_evidence,
    rag_result
)

print("=" * 70)
print("GEMINI POS ROOT CAUSE INVESTIGATION")
print("=" * 70)

print()

print(
    "Transaction:",
    txn006_gemini_result[
        "transaction_id"
    ]
)

print(
    "Evidence Status:",
    txn006_gemini_result[
        "evidence_status"
    ]
)

print(
    "Model:",
    txn006_gemini_result[
        "model"
    ]
)

print()

print(
    txn006_gemini_result[
        "generated_analysis"
    ]
)

GEMINI POS ROOT CAUSE INVESTIGATION

Transaction: TXN006
Evidence Status: HISTORICAL_CORRELATION
Model: gemini-3.6-flash

OBSERVATION:
Transaction TXN006 on store STORE103, lane LANE03 experienced a HIGH_CASHBACK anomaly. The transaction had an expected cashback amount of $5.00 and an actual cashback amount of $30.00, resulting in a cashback variance of $25.00.

SUPPORTING EVIDENCE:
- Current Transaction Evidence: Analysis of TXN006 shows no correlation within the current transaction facts (Matched Values: []).
- Historical Evidence: Historical correlation was identified on LANE03. Previous gift card failure events occurred on this lane: TXN004 (sequence 1, failure amount $10.00) and TXN005 (sequence 2, failure amount $15.00).
- Matching Amounts: The accumulated historical failure amount of $25.00 ($10.00 + $15.00) matches the $25.00 cashback variance in TXN006. 

ROOT-CAUSE HYPOTHESIS:
It is hypothesized that transaction-specific state from previous failed gift card attempts (TXN004 a

Very important second test: TXN003

In [81]:
txn003_rag_result = rag_agent(
    txn003_evidence
)

In [82]:
txn003_gemini_result = gemini_rca_agent(
    txn003_evidence,
    txn003_rag_result
)

print("=" * 70)
print("GEMINI INCONCLUSIVE CASE TEST")
print("=" * 70)

print()

print(
    "Transaction:",
    txn003_gemini_result[
        "transaction_id"
    ]
)

print(
    "Evidence Status:",
    txn003_gemini_result[
        "evidence_status"
    ]
)

print()

print(
    txn003_gemini_result[
        "generated_analysis"
    ]
)

GEMINI INCONCLUSIVE CASE TEST

Transaction: TXN003
Evidence Status: INSUFFICIENT_EVIDENCE

OBSERVATION:
Transaction TXN003 at Store STORE102, Lane LANE01 encountered a NEGATIVE_CASHBACK anomaly. The expected cashback was $5.00, but the actual cashback returned was $-15.00, resulting in a calculated cashback variance of $-20.00.

SUPPORTING EVIDENCE:
- Current Transaction Analysis: Correlation Found is False (Matched Values: []). The cashback variance of $-20.00 does not correlate with any identified values within the current transaction.
- Historical Analysis: Historical Correlation is False. There are no previous gift card failure events recorded ([]), and the accumulated historical failure amount is $0.00.
- Evidence Status: The overall evidence status is explicitly marked as INSUFFICIENT_EVIDENCE. No matching monetary amounts were found in either current or historical transaction data.

ROOT-CAUSE HYPOTHESIS:
INCONCLUSIVE. 
Because no monetary correlations were found in the current 

Part 13 — LangGraph Multi-Agent Workflow

## Part 13: LangGraph Multi-Agent Orchestration

LangGraph is used to orchestrate the POS anomaly investigation workflow.

The workflow coordinates:

1. Transaction Health Agent
2. Pattern Analysis Agent
3. Historical Context Agent
4. RAG Knowledge Agent
5. Gemini RCA Agent
6. Human Review

Normal transactions exit early, while anomalous transactions continue
through the investigation workflow.

The final Gemini-generated RCA is treated as a hypothesis and requires
human engineering review.

Install/import LangGraph

In [83]:
!pip -q install -U langgraph

In [84]:
from typing import TypedDict, Any
from langgraph.graph import StateGraph, START, END

print("LangGraph imported successfully")

LangGraph imported successfully


Define the shared state

In [85]:
class POSInvestigationState(TypedDict, total=False):

    transaction: dict
    transaction_history: list

    health_result: dict
    pattern_evidence: dict
    historical_evidence: dict

    evidence_status: str

    rag_result: dict
    rca_result: dict

    human_review: str
    error: str

Transaction Health Node

In [86]:
def health_node(state):

    transaction = state["transaction"]

    result = check_transaction_health(
        transaction
    )

    print(
        "[Health Agent]",
        transaction["transaction_id"],
        "->",
        result["anomaly_type"]
    )

    return {
        "health_result": result
    }

Router

In [87]:
def anomaly_router(state):

    health = state["health_result"]

    if health["status"] == "NORMAL":
        return "normal"

    return "investigate"

Pattern Analysis Node

In [88]:
def pattern_node(state):

    result = pattern_analysis_agent(
        state["transaction"],
        state["health_result"]
    )

    print(
        "[Pattern Agent]",
        "Correlation:",
        result["correlation_found"]
    )

    return {
        "pattern_evidence": result
    }

Historical Context Node

In [89]:
def history_node(state):

    result = historical_context_agent(
        state["transaction"],
        state["transaction_history"]
    )

    print(
        "[Historical Agent]",
        "Correlation:",
        result[
            "analysis"
        ]["historical_correlation"]
    )

    return {
        "historical_evidence": result
    }

Evidence Aggregator

In [90]:
def evidence_node(state):

    pattern = state[
        "pattern_evidence"
    ]

    history = state[
        "historical_evidence"
    ]

    if pattern["correlation_found"]:

        status = (
            "CURRENT_TRANSACTION_CORRELATION"
        )

    elif history[
        "analysis"
    ]["historical_correlation"]:

        status = (
            "HISTORICAL_CORRELATION"
        )

    else:

        status = (
            "INSUFFICIENT_EVIDENCE"
        )

    print(
        "[Evidence Agent]",
        status
    )

    return {
        "evidence_status": status
    }

RAG Node

In [91]:
def rag_node(state):

    investigation_result = {

        "transaction":
            state["transaction"],

        "health_result":
            state["health_result"],

        "pattern_evidence":
            state["pattern_evidence"],

        "historical_evidence":
            state["historical_evidence"],

        "evidence_status":
            state["evidence_status"]
    }

    result = rag_agent(
        investigation_result
    )

    print("[RAG Agent] Retrieved:")

    for doc in result[
        "retrieved_documents"
    ]:

        print(
            " -",
            doc["title"]
        )

    return {
        "rag_result": result
    }

Gemini RCA Node

In [92]:
def gemini_node(state):

    investigation_result = {

        "transaction":
            state["transaction"],

        "health_result":
            state["health_result"],

        "pattern_evidence":
            state["pattern_evidence"],

        "historical_evidence":
            state["historical_evidence"],

        "evidence_status":
            state["evidence_status"]
    }

    try:

        result = gemini_rca_agent(
            investigation_result,
            state["rag_result"]
        )

        print(
            "[Gemini RCA Agent]",
            "Analysis generated"
        )

        return {
            "rca_result": result
        }

    except Exception as e:

        print(
            "[Gemini RCA Agent] ERROR:",
            str(e)
        )

        return {
            "rca_result": {
                "generated_analysis":
                    "Gemini RCA unavailable.",

                "requires_human_review":
                    True
            },

            "error":
                str(e)
        }

Human Review Node

In [93]:
def human_review_node(state):

    print(
        "[Human Review]",
        "Engineering review required"
    )

    return {
        "human_review":
            "PENDING_ENGINEERING_REVIEW"
    }

Build the Graph

In [94]:
workflow = StateGraph(
    POSInvestigationState
)

In [95]:
workflow.add_node(
    "health_agent",
    health_node
)

workflow.add_node(
    "pattern_agent",
    pattern_node
)

workflow.add_node(
    "history_agent",
    history_node
)

workflow.add_node(
    "evidence_agent",
    evidence_node
)

workflow.add_node(
    "rag_agent",
    rag_node
)

workflow.add_node(
    "gemini_rca_agent",
    gemini_node
)

workflow.add_node(
    "human_review",
    human_review_node
)

Add the edges

In [96]:
workflow.add_edge(
    START,
    "health_agent"
)

In [97]:
workflow.add_conditional_edges(
    "health_agent",
    anomaly_router,
    {
        "normal": END,
        "investigate": "pattern_agent"
    }
)

In [98]:
workflow.add_edge(
    "pattern_agent",
    "history_agent"
)

workflow.add_edge(
    "history_agent",
    "evidence_agent"
)

workflow.add_edge(
    "evidence_agent",
    "rag_agent"
)

workflow.add_edge(
    "rag_agent",
    "gemini_rca_agent"
)

workflow.add_edge(
    "gemini_rca_agent",
    "human_review"
)

workflow.add_edge(
    "human_review",
    END
)

In [99]:
pos_investigation_graph = (
    workflow.compile()
)

print(
    "POS Investigation LangGraph compiled successfully"
)

POS Investigation LangGraph compiled successfully


Test a NORMAL transaction first

In [100]:
normal_result = (
    pos_investigation_graph.invoke(
        {
            "transaction":
                transactions[0],

            "transaction_history":
                all_transactions
        }
    )
)

[Health Agent] TXN001 -> NONE


In [101]:
print(
    normal_result[
        "health_result"
    ]
)

{'transaction_id': 'TXN001', 'status': 'NORMAL', 'anomaly_type': 'NONE', 'expected_cashback': 5.0, 'actual_cashback': 5.0, 'variance': 0.0}


Run the complete TXN006 workflow

In [102]:
final_result = (
    pos_investigation_graph.invoke(
        {
            "transaction":
                historical_transactions[2],

            "transaction_history":
                all_transactions
        }
    )
)

[Health Agent] TXN006 -> HIGH_CASHBACK
[Pattern Agent] Correlation: False
[Historical Agent] Correlation: True
[Evidence Agent] HISTORICAL_CORRELATION
[RAG Agent] Retrieved:
 - Cashback Anomaly Investigation Guide
 - POS Transaction State Management Guide
[Gemini RCA Agent] Analysis generated
[Human Review] Engineering review required


Display the final result

In [103]:
print("=" * 70)
print("AI POS TRANSACTION ANOMALY INVESTIGATION")
print("=" * 70)

print()

print(
    "Transaction:",
    final_result[
        "transaction"
    ]["transaction_id"]
)

print(
    "Anomaly:",
    final_result[
        "health_result"
    ]["anomaly_type"]
)

print(
    "Cashback Variance: $",
    final_result[
        "health_result"
    ]["variance"]
)

print(
    "Evidence Status:",
    final_result[
        "evidence_status"
    ]
)

print()

print("=" * 70)
print("GEMINI ROOT CAUSE INVESTIGATION")
print("=" * 70)

print(
    final_result[
        "rca_result"
    ]["generated_analysis"]
)

print()

print("=" * 70)
print(
    "Human Review:",
    final_result[
        "human_review"
    ]
)

AI POS TRANSACTION ANOMALY INVESTIGATION

Transaction: TXN006
Anomaly: HIGH_CASHBACK
Cashback Variance: $ 25.0
Evidence Status: HISTORICAL_CORRELATION

GEMINI ROOT CAUSE INVESTIGATION
OBSERVATION:
Transaction TXN006 occurred at Store STORE103 on Lane LANE03 and triggered a HIGH_CASHBACK anomaly. The expected cashback amount was $5.00, while the actual cashback dispensed was $30.00, resulting in a calculated cashback variance of $25.00.

SUPPORTING EVIDENCE:
- Current Transaction Analysis: No monetary correlation was found within the current transaction data (matched values list is empty).
- Historical Analysis: A historical monetary correlation was identified on the same lane (LANE03). Previous gift card failure events occurred on this lane: TXN004 (sequence 1, $10.00 failure) and TXN005 (sequence 2, $15.00 failure).
- Matching Monetary Amount: The accumulated historical failure amount from TXN004 and TXN005 totals $25.00 ($10.00 + $15.00), which directly matches the $25.00 cashback va

Part 14 — Evaluation

## Part 14: Evaluation

The AI POS Transaction Anomaly Investigation Assistant is evaluated
using synthetic POS transaction scenarios.

The evaluation verifies:

- Correct anomaly detection
- Correct workflow routing
- Current-transaction correlation detection
- Historical correlation detection
- RAG retrieval
- Gemini RCA generation
- Inconclusive handling when evidence is insufficient
- Human-review requirement

The evaluation uses synthetic scenarios and does not represent
production accuracy.

Define the test cases

In [104]:
evaluation_cases = [
    {
        "transaction": transactions[0],   # TXN001
        "expected_anomaly": "NONE",
        "expected_evidence": None
    },

    {
        "transaction": transactions[1],   # TXN002
        "expected_anomaly": "HIGH_CASHBACK",
        "expected_evidence":
            "CURRENT_TRANSACTION_CORRELATION"
    },

    {
        "transaction": transactions[2],   # TXN003
        "expected_anomaly": "NEGATIVE_CASHBACK",
        "expected_evidence":
            "INSUFFICIENT_EVIDENCE"
    },

    {
        "transaction": historical_transactions[2],  # TXN006
        "expected_anomaly": "HIGH_CASHBACK",
        "expected_evidence":
            "HISTORICAL_CORRELATION"
    }
]

print(
    "Evaluation cases:",
    len(evaluation_cases)
)

Evaluation cases: 4


Run all cases through the final LangGraph

In [105]:
evaluation_results = []

for case in evaluation_cases:

    txn = case["transaction"]

    print("\n" + "=" * 70)
    print("Evaluating:", txn["transaction_id"])
    print("=" * 70)

    result = pos_investigation_graph.invoke(
        {
            "transaction": txn,
            "transaction_history": all_transactions
        }
    )

    actual_anomaly = (
        result["health_result"]["anomaly_type"]
    )

    actual_evidence = (
        result.get("evidence_status")
    )

    anomaly_correct = (
        actual_anomaly ==
        case["expected_anomaly"]
    )

    evidence_correct = (
        actual_evidence ==
        case["expected_evidence"]
    )

    evaluation_results.append({
        "Transaction":
            txn["transaction_id"],

        "Expected Anomaly":
            case["expected_anomaly"],

        "Detected Anomaly":
            actual_anomaly,

        "Anomaly Correct":
            anomaly_correct,

        "Expected Evidence":
            case["expected_evidence"]
            if case["expected_evidence"]
            else "NOT_REQUIRED",

        "Detected Evidence":
            actual_evidence
            if actual_evidence
            else "NOT_REQUIRED",

        "Evidence Correct":
            evidence_correct
    })


Evaluating: TXN001
[Health Agent] TXN001 -> NONE

Evaluating: TXN002
[Health Agent] TXN002 -> HIGH_CASHBACK
[Pattern Agent] Correlation: True
[Historical Agent] Correlation: False
[Evidence Agent] CURRENT_TRANSACTION_CORRELATION
[RAG Agent] Retrieved:
 - Cashback Anomaly Investigation Guide
 - Gift Card Failure Handling Guide
[Gemini RCA Agent] Analysis generated
[Human Review] Engineering review required

Evaluating: TXN003
[Health Agent] TXN003 -> NEGATIVE_CASHBACK
[Pattern Agent] Correlation: False
[Historical Agent] Correlation: False
[Evidence Agent] INSUFFICIENT_EVIDENCE
[RAG Agent] Retrieved:
 - Cashback Anomaly Investigation Guide
 - POS Transaction State Management Guide
[Gemini RCA Agent] Analysis generated
[Human Review] Engineering review required

Evaluating: TXN006
[Health Agent] TXN006 -> HIGH_CASHBACK
[Pattern Agent] Correlation: False
[Historical Agent] Correlation: True
[Evidence Agent] HISTORICAL_CORRELATION
[RAG Agent] Retrieved:
 - Cashback Anomaly Investigation G

Display the evaluation table

In [106]:
evaluation_df = pd.DataFrame(
    evaluation_results
)

display(evaluation_df)

,Transaction,Expected Anomaly,Detected Anomaly,Anomaly Correct,Expected Evidence,Detected Evidence,Evidence Correct
0,TXN001,NONE,NONE,True,NOT_REQUIRED,NOT_REQUIRED,True
1,TXN002,HIGH_CASHBACK,HIGH_CASHBACK,True,CURRENT_TRANSACTION_CORRELATION,CURRENT_TRANSACTION_CORRELATION,True
2,TXN003,NEGATIVE_CASHBACK,NEGATIVE_CASHBACK,True,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,True
3,TXN006,HIGH_CASHBACK,HIGH_CASHBACK,True,HISTORICAL_CORRELATION,HISTORICAL_CORRELATION,True


Calculate controlled test accuracy

In [107]:
anomaly_accuracy = (
    evaluation_df[
        "Anomaly Correct"
    ].mean() * 100
)

evidence_accuracy = (
    evaluation_df[
        "Evidence Correct"
    ].mean() * 100
)

print(
    f"Anomaly Detection Accuracy: "
    f"{anomaly_accuracy:.1f}%"
)

print(
    f"Evidence Routing Accuracy: "
    f"{evidence_accuracy:.1f}%"
)

Anomaly Detection Accuracy: 100.0%
Evidence Routing Accuracy: 100.0%


Part 15 — Final Demo

## Part 15: Final Capstone Demonstration

The final demonstration executes the complete multi-agent investigation
workflow for a selected POS transaction and presents the anomaly,
supporting evidence, retrieved knowledge, GenAI RCA hypothesis and
human-review status.

In [108]:
def run_pos_investigation(transaction):

    print("=" * 72)
    print(" AI POS TRANSACTION ANOMALY INVESTIGATION ASSISTANT")
    print("=" * 72)

    print(
        "\nTransaction:",
        transaction["transaction_id"]
    )

    print(
        "Store:",
        transaction["store_id"]
    )

    print(
        "Lane:",
        transaction["lane_id"]
    )

    print("\nRunning investigation...\n")

    result = pos_investigation_graph.invoke(
        {
            "transaction":
                transaction,

            "transaction_history":
                all_transactions
        }
    )

    health = result["health_result"]

    print("\n" + "=" * 72)
    print(" INVESTIGATION SUMMARY")
    print("=" * 72)

    print(
        "\nAnomaly Type:",
        health["anomaly_type"]
    )

    print(
        "Expected Cashback: $",
        health["expected_cashback"]
    )

    print(
        "Actual Cashback: $",
        health["actual_cashback"]
    )

    print(
        "Variance: $",
        health["variance"]
    )

    # Normal transaction
    if health["status"] == "NORMAL":

        print(
            "\nResult: Normal transaction."
        )

        print(
            "No GenAI investigation required."
        )

        return result

    print(
        "\nEvidence Status:",
        result["evidence_status"]
    )

    print("\nRetrieved Knowledge:")

    for doc in result[
        "rag_result"
    ]["retrieved_documents"]:

        print(
            " -",
            doc["title"]
        )

    print("\n" + "=" * 72)
    print(" GEMINI RCA")
    print("=" * 72)

    print(
        result[
            "rca_result"
        ]["generated_analysis"]
    )

    print("\n" + "=" * 72)

    print(
        "Human Review:",
        result["human_review"]
    )

    print("=" * 72)

    return result

In [109]:
demo_result = run_pos_investigation(
    historical_transactions[2]
)

 AI POS TRANSACTION ANOMALY INVESTIGATION ASSISTANT

Transaction: TXN006
Store: STORE103
Lane: LANE03

Running investigation...

[Health Agent] TXN006 -> HIGH_CASHBACK
[Pattern Agent] Correlation: False
[Historical Agent] Correlation: True
[Evidence Agent] HISTORICAL_CORRELATION
[RAG Agent] Retrieved:
 - Cashback Anomaly Investigation Guide
 - POS Transaction State Management Guide
[Gemini RCA Agent] Analysis generated
[Human Review] Engineering review required

 INVESTIGATION SUMMARY

Anomaly Type: HIGH_CASHBACK
Expected Cashback: $ 5.0
Actual Cashback: $ 30.0
Variance: $ 25.0

Evidence Status: HISTORICAL_CORRELATION

Retrieved Knowledge:
 - Cashback Anomaly Investigation Guide
 - POS Transaction State Management Guide

 GEMINI RCA
OBSERVATION:
Transaction TXN006 at STORE103 on LANE03 triggered a HIGH_CASHBACK anomaly. The key monetary values recorded for this transaction are:
- Expected Cashback: $5.00
- Actual Cashback: $30.00
- Cashback Variance: $25.00

SUPPORTING EVIDENCE:
- Curr